# Kazakh Profanity Detection — DSSC5113 Bonus Project

**Author:** Zhanabil Alikhan · **Student ID:** 25MD0222 · KBTU, Fall 2026

**Task.** Input: one Kazakh product review / comment. Output: **0 = no Kazakh profanity**, **1 = Kazakh profanity present**, plus the model's probability.

**How to run**
- **Google Colab:** open this notebook, then *Runtime → Run all*. All files are created next to the notebook.
- **Local:** `pip install -r requirements.txt` then `jupyter notebook Kazakh_Profanity_Detection.ipynb` → *Run all*.
- Only want predictions? Run **Step 0** and then jump to **Step 7** (loads the saved model, no retraining).

**Pipeline**

| Step | What happens | Output files |
|---|---|---|
| 0 | Setup, seed, helper module | `src/textprep.py`, `src/lexicon.py` |
| 1 | Download KazSAnDRA reviews | `data/all_reviews.csv` (not on GitHub) |
| 2 | Find profanity candidates, (manual) labeling, synthetic examples | `data/to_review.csv`, `data/labeled.csv` |
| 3 | Remove duplicates, group near-duplicates, train/val/test split | `data/splits/*.csv` |
| 4 | Train TF-IDF char n-gram + Logistic Regression, tune on validation | `models/profanity_model.joblib`, `models/config.json` |
| 5 | Evaluate **once** on test | `results/metrics.json`, `results/confusion_matrix.png` |
| 6 | Error analysis | `results/errors.csv`, `results/sample_predictions.csv` |
| 7 | Predict on your own text | `src/predict.py` |
| 8 | Write README, requirements, zip project | `README.md`, `requirements.txt` |

## Step 0 — Setup
Install packages, fix the random seed, create the folders.

In [1]:
%pip install -q datasets pandas scikit-learn matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, json, random, platform
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

for folder in ["data/splits", "src", "models", "results"]:
    os.makedirs(folder, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| seed", SEED)

Python 3.13.9 | pandas 2.3.3 | seed 42


### Text preprocessing module
Saved as a `.py` file so the saved model can be loaded later (in `predict.py` or another notebook) without this notebook.

- `remove_personal_info` — removes URLs, e-mails, phone numbers and @usernames (privacy; done once during data preparation).
- `normalize` — used **inside the model**: lowercase, `ё→е`, digits→`0`, mask symbols (`* @ # $ %`) → `*`, repeated letters (`бооооқ`) → one letter, punctuation → space, long texts cut to 2000 characters.

In [3]:
%%writefile src/textprep.py
import re

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE_RE = re.compile(r"\+?\d[\d\-\s()]{7,}\d")
MENTION_RE = re.compile(r"@\w+")

MAX_CHARS = 2000  # long comments are cut (profanity is almost always visible early)


def remove_personal_info(text: str) -> str:
    # Remove links, e-mails, phone numbers and @usernames.
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = PHONE_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize(text: str) -> str:
    # Normalization applied to every comment before it becomes model input.
    t = str(text).lower()[:MAX_CHARS]
    t = t.replace("ё", "е")
    t = re.sub(r"[*@#$%]+", "*", t)          # masking symbols -> one '*'
    t = re.sub(r"\d", "0", t)                 # all digits -> 0
    t = re.sub(r"(\w)\1{2,}", r"\1", t)       # бооооқ -> боқ
    t = re.sub(r"[^\w\s*]", " ", t)           # punctuation -> space (keep '*')
    t = re.sub(r"_", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def dedup_key(text: str) -> str:
    # Aggressive key used to detect near-duplicates (only letters).
    return re.sub(r"[^a-zа-яәіңғүұқөһ]", "", normalize(text))

Writing src/textprep.py


### Profanity lexicon (used only to *find candidates* and pre-label them)
The lexicon is **not** the model. It helps find rare profane reviews among 180k reviews and gives a first label that a human checks.
Patterns work on normalized text and include Cyrillic, Latin and masked spellings. Word boundaries avoid false hits like *көтеру* (lift), *амал* (way), *бокс* (boxing).

In [4]:
%%writefile src/lexicon.py
import re
from textprep import normalize

PATTERNS = {
    "sik":        r"\bс[іи]к(?:ті|ти|тім|тим|тің|тір|тир|ейін|ейин|кен|іп|ип|у)?\b",
    "sigeyin":    r"\bс[іи]гей[іи]н\b",
    "qotaq":      r"\b[қк]ота[қк]\w*",
    "boq":        r"\bбоқ\w*|\bбок(?:тай|ты|тан)\b",
    "kot":        r"\bкөт(?:ің|іңді|іңе|ке|тен|ім|ін|іне)?\b",
    "am":         r"\bам(?:ың|ыңды|ыңа|ыңның|ыңнан|ын|ына)\b",
    "zhezokshe":  r"\b[жз]ез[өо]кше\w*",
    "curse_auz":  r"\b[әа]кең(?:н[іи]ң)?\s+аузын\b|\bаузын\s+[ұу]райын\b",
    "it_balasy":  r"\bиттің\s+баласы\b",
    "latin":      r"\b(?:sik(?:ti|tim|tin|eyin)?|sigeyin|qotaq|kotak|boq|zhezokshe|jezokshe|amyndy)\b",
    "masked":     r"\bс\*+[кг]\w*|\b[қк]\*+та[қк]\w*|\b[қк]о\*+\w*|\bб\*+[қк]\w*|\bбо\*+\w*|\bк\*+т[іi]\w*",
}
COMPILED = {name: re.compile(p) for name, p in PATTERNS.items()}


def lexicon_hits(text: str) -> list:
    t = normalize(text)
    return [name for name, rx in COMPILED.items() if rx.search(t)]


def lexicon_label(text: str) -> int:
    return int(len(lexicon_hits(text)) > 0)

Writing src/lexicon.py


In [5]:
import sys, importlib
if "src" not in sys.path:
    sys.path.insert(0, "src")
import textprep, lexicon
importlib.reload(textprep); importlib.reload(lexicon)
from textprep import normalize, remove_personal_info, dedup_key
from lexicon import lexicon_hits, lexicon_label

# quick check of the helpers
for s in ["Тауар ӨТЕ жақсы!!! Рахмет :)", "Бооооқ телефон", "Ауыр екен, көтеру қиын", "с**к осы дүкенді",
          "Маған жазыңыз +7 701 123 45 67 немесе test@mail.kz"]:
    print(f"{s!r:55} -> normalized={normalize(s)!r:35} hits={lexicon_hits(s)}")
print(remove_personal_info("Маған жазыңыз +7 701 123 45 67 немесе test@mail.kz"))

'Тауар ӨТЕ жақсы!!! Рахмет :)'                          -> normalized='тауар өте жақсы рахмет'            hits=[]
'Бооооқ телефон'                                        -> normalized='боқ телефон'                       hits=['boq']
'Ауыр екен, көтеру қиын'                                -> normalized='ауыр екен көтеру қиын'             hits=[]
'с**к осы дүкенді'                                      -> normalized='с*к осы дүкенді'                   hits=['masked']
'Маған жазыңыз +7 701 123 45 67 немесе test@mail.kz'    -> normalized='маған жазыңыз 0 0 0 00 00 немесе test*mail kz' hits=[]
Маған жазыңыз немесе


## Step 1 — Download Kazakh reviews (KazSAnDRA)
**Source:** KazSAnDRA — *Kazakh Sentiment Analysis Dataset of Reviews and Attitudes* (ISSAI, Nazarbayev University), ~180k real reviews of apps, products, books and places, license CC BY 4.0.
Hugging Face: https://huggingface.co/datasets/issai/kazsandra · GitHub: https://github.com/IS2AI/KazSAnDRA

**Why it fits:** these are real user reviews written in Kazakh (often mixed with Russian, Latin script, slang) — the same kind of text a marketplace would moderate. Its labels are *sentiment*, **not profanity**, so we do not use them as our labels (a negative review is not automatically profane).

If `data/all_reviews.csv` already exists, the download is skipped. This file is large, so it is in `.gitignore`.

In [7]:
RAW_PATH = "data/all_reviews.csv"

if os.path.exists(RAW_PATH):
    print("Found", RAW_PATH, "- skipping download")
else:
    from datasets import load_dataset
    try:
        ds = load_dataset("issai/kazsandra", "full")
    except Exception as e:
        print("Config 'full' failed, trying default:", e)
        ds = load_dataset("issai/kazsandra")
    frames = []
    for split in ds:
        d = ds[split].to_pandas()
        d["orig_split"] = split
        frames.append(d)
    raw = pd.concat(frames, ignore_index=True)
    print("Original columns:", list(raw.columns))

    # pick the column with the original (uncleaned) review text
    text_col = next((c for c in ["text", "review", "content", "text_cleaned"] if c in raw.columns),
                    raw.select_dtypes("object").columns[0])
    keep = {text_col: "text"}
    if "domain" in raw.columns:
        keep["domain"] = "domain"
    raw = raw[list(keep)].rename(columns=keep)
    if "domain" not in raw.columns:
        raw["domain"] = "unknown"
    raw.to_csv(RAW_PATH, index=False, encoding="utf-8")
    print("Saved", len(raw), "reviews to", RAW_PATH)

reviews = pd.read_csv(RAW_PATH)
reviews["text"] = reviews["text"].astype(str).map(remove_personal_info)   # privacy
reviews = reviews[reviews["text"].str.len() > 0].reset_index(drop=True)
print("Reviews after cleaning:", len(reviews))
reviews.sample(5, random_state=SEED)

Config 'full' failed, trying default: Dataset 'issai/kazsandra' is a gated dataset on the Hub. You must be authenticated to access it.


DatasetNotFoundError: Dataset 'issai/kazsandra' is a gated dataset on the Hub. You must be authenticated to access it.

## Step 2 — Build the labeled dataset

### Labeling rules (Kazakh profanity)
| Case | Label | Example |
|---|---|---|
| Obscene / vulgar Kazakh words (sexual, genital, excretory) | **1** | *боқ*, *қотақ*, *сік-*, *жезөкше*, *көтіңе* |
| Fixed Kazakh curse formulas | **1** | *әкеңнің аузын …*, *иттің баласы* |
| Masked or misspelled forms whose meaning is clear | **1** | *с\*к*, *б\*\*қ*, *бооооқ*, *qotaq* (Latin) |
| Kazakh profanity inside a mixed Kazakh–Russian comment | **1** | *доставка ужас, телефон боқ* |
| Negative / angry review **without** profanity | **0** | *Сапасы өте нашар, ақшамды қайтарыңыз* |
| Mild insults (not obscene) | **0** | *ақымақ*, *топас*, *оңбаған* |
| Words that only *contain* a profane-looking substring | **0** | *көтеру*, *амал*, *бокс* |
| Profanity only in Russian (no Kazakh profanity) | **0** (out of task scope; noted as limitation) | — |
| Truly unclear meaning | **0**, written in `note` column | — |

### How labels are produced
1. **Real reviews:** a random sample of 2,000 KazSAnDRA reviews + every review where the lexicon finds a candidate. Each gets an `auto_label` from the lexicon.
2. **Manual check:** open `data/to_review.csv` in Excel/Google Sheets, fill the empty `label` column (0/1) using the rules above (start from `auto_label`), save as **`data/manual_labels.csv`** (UTF-8 CSV) and re-run from this step. Rows without a manual label keep the lexicon label (`label_source = lexicon_auto`).
3. **Synthetic examples:** real profanity is rare, so we add template-generated comments (column `is_synthetic = 1`). Every positive template has a matching **clean negative** template with the same structure, so the model must learn the profane word, not the sentence shape.

In [ ]:
N_RANDOM = 2000
MAX_CANDIDATES = 1500

reviews["hits"] = reviews["text"].map(lexicon_hits)
candidates = reviews[reviews["hits"].str.len() > 0]
print("Lexicon candidates in the whole dataset:", len(candidates))

random_part = reviews.sample(n=min(N_RANDOM, len(reviews)), random_state=SEED)
cand_part = candidates.sample(n=min(MAX_CANDIDATES, len(candidates)), random_state=SEED)
real = pd.concat([random_part, cand_part]).drop_duplicates(subset="text").reset_index(drop=True)
real["auto_label"] = (real["hits"].str.len() > 0).astype(int)
real["matched_rules"] = real["hits"].map(lambda h: ",".join(h))
real["review_id"] = ["r%05d" % i for i in range(len(real))]

to_review = real[["review_id", "text", "domain", "matched_rules", "auto_label"]].copy()
to_review["label"] = ""      # <- fill by hand (0/1)
to_review["note"] = ""
to_review.to_csv("data/to_review.csv", index=False, encoding="utf-8-sig")
print("Saved data/to_review.csv with", len(to_review), "rows (", to_review.auto_label.sum(), "auto-positive )")
to_review[to_review.auto_label == 1].head(10)

In [ ]:
MANUAL_PATH = "data/manual_labels.csv"
real["label"] = real["auto_label"]
real["label_source"] = "lexicon_auto"
real["note"] = ""

if os.path.exists(MANUAL_PATH):
    manual = pd.read_csv(MANUAL_PATH, encoding="utf-8-sig")
    manual = manual[["review_id", "label", "note"]].rename(columns={"label": "manual_label", "note": "manual_note"})
    real = real.merge(manual, on="review_id", how="left")
    has_manual = real["manual_label"].notna() & (real["manual_label"].astype(str).str.strip() != "")
    real.loc[has_manual, "label"] = real.loc[has_manual, "manual_label"].astype(float).astype(int)
    real.loc[has_manual, "label_source"] = "manual"
    real.loc[has_manual, "note"] = real.loc[has_manual, "manual_note"].fillna("")
    changed = (real["label"] != real["auto_label"]).sum()
    print(f"Manual labels used for {has_manual.sum()} rows; {changed} lexicon labels were corrected by hand")
else:
    print("WARNING: data/manual_labels.csv not found -> real reviews use lexicon labels only.")
    print("         Check them by hand for a better project (see instructions above).")

real["is_synthetic"] = 0
real["group"] = real["text"].map(dedup_key)       # near-duplicate group
print(real["label_source"].value_counts().to_dict(), "| class counts:", real["label"].value_counts().to_dict())

### Synthetic examples (clearly marked `is_synthetic = 1`)
Written by the author with templates. Every template is a near-duplicate family, so all comments from one *(template, word)* pair go to the **same split** (`group` column).

In [ ]:
rng = random.Random(SEED)
products = ["Телефон", "Бұл тауар", "Кроссовка", "Наушник", "Сөмке", "Зарядка", "Кітап",
            "Көйлек", "Шаңсорғыш", "Тоңазытқыш", "Сағат", "Планшет"]
bad_words = ["боқ", "қотақ", "б*қ", "қ*тақ", "boq", "qotaq", "бооооқ", "БОҚ", "қотақбас", "боқтай"]
curse_phrases = ["сіктім осы дүкенді", "әкеңнің аузын ұрайын", "жезөкше сатушы алдады",
                 "көтіңе тық мұндай сапаны", "с*ктім бұл тауарды", "иттің баласы курьер",
                 "sikteyin osy magazindi", "амыңды с*гейін", "әкеңнің аузын, ақшамды қайтар", "zhezokshe satushy"]

pos_word_templates = [
    "{p} нағыз {w}, ақшамды босқа төледім", "{w} сияқты сапа, {p} екі күнде бұзылды",
    "Сатушы {w}, {p} суреттегідей емес", "{p} {w} екен, ешкімге ұсынбаймын",
    "Курьер {w}, {p} үш апта келді", "Мынау {p} емес, {w}!!!",
    "{p} отстой, {w} ғой мынау", "доставка ужас, {p} {w}",
]
pos_phrase_templates = ["{p} сапасыз, {c}", "{c}! {p} бірінші күні сынды",
                        "{p} жарамайды. {c}", "Ақшамды қайтарыңдар, {c}"]
neg_templates = [
    "{p} сапасы өте нашар, ақшамды босқа төледім", "Нашар сапа, {p} екі күнде бұзылды",
    "Сатушы алдады, {p} суреттегідей емес", "{p} жарамсыз екен, ешкімге ұсынбаймын",
    "Курьер дөрекі, {p} үш апта келді", "Мынау {p} емес, алдау!!!",
    "{p} отстой, ақша лақтыру ғой мынау", "доставка ужас, {p} сапасыз",
    "{p} ауыр екен, көтеру қиын", "Амал жоқ, {p} қайтардым", "{p} жаман емес, бірақ бағасы қымбат",
    "{p} ақымақ дизайн, бірақ жұмыс істейді", "Боксқа арналған {p} жақсы, рахмет",
    "Мысық {p} сындырды, бірақ тауар жақсы", "{p} сапасыз, ақшамды қайтарыңдар!",
    "Ақшамды қайтарыңдар, сатушы жауап бермейді", "{p} керемет, бәріне ұсынамын", "Тез жеткізді, {p} ұнады",
]

synth = []
for ti, t in enumerate(pos_word_templates):
    for w in bad_words:
        for p in rng.sample(products, 3):
            synth.append((t.format(p=p, w=w), 1, f"pw{ti}_{w}"))
for ti, t in enumerate(pos_phrase_templates):
    for c in curse_phrases:
        for p in rng.sample(products, 3):
            synth.append((t.format(p=p, c=c), 1, f"pp{ti}_{c}"))
for ti, t in enumerate(neg_templates):
    for p in products:
        synth.append((t.format(p=p), 0, f"n{ti}"))

synth = pd.DataFrame(synth, columns=["text", "label", "group"])
synth["group"] = "synth_" + synth["group"]
synth["is_synthetic"] = 1
synth["label_source"] = "synthetic_template"
synth["domain"] = "synthetic"
synth["note"] = ""
print("Synthetic examples:", len(synth), "| class counts:", synth.label.value_counts().to_dict())
synth.sample(8, random_state=SEED)

In [ ]:
cols = ["text", "label", "is_synthetic", "label_source", "domain", "group", "note"]
data = pd.concat([real[cols], synth[cols]], ignore_index=True)

# remove exact duplicates (same normalized text); if labels disagree keep the manual/real one
before = len(data)
data["norm"] = data["text"].map(normalize)
data = data.drop_duplicates(subset="norm", keep="first").drop(columns="norm").reset_index(drop=True)
print(f"Removed {before - len(data)} exact duplicates")

data.to_csv("data/labeled.csv", index=False, encoding="utf-8")
print("Total samples:", len(data))
print(pd.crosstab(data["is_synthetic"].map({0: "real", 1: "synthetic"}), data["label"], margins=True))

## Step 3 — Train / validation / test split
- Split **by group**, not by row: a real review's group is its letters-only text (catches near-duplicates that differ in punctuation, case, digits or repeated letters); a synthetic group is its *(template, word)* family. A whole group always goes to one split → no leakage.
- 70 / 15 / 15, stratified by label, seed = 42.

In [ ]:
from sklearn.model_selection import train_test_split

groups = data.groupby("group")["label"].max().reset_index()
g_train, g_tmp = train_test_split(groups, test_size=0.30, stratify=groups["label"], random_state=SEED)
g_val, g_test = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp["label"], random_state=SEED)

split_of = {**{g: "train" for g in g_train.group}, **{g: "val" for g in g_val.group}, **{g: "test" for g in g_test.group}}
data["split"] = data["group"].map(split_of)
train_df, val_df, test_df = (data[data.split == s].reset_index(drop=True) for s in ["train", "val", "test"])

assert not (set(train_df.group) & set(test_df.group)) and not (set(train_df.group) & set(val_df.group))
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(f"data/splits/{name}.csv", index=False, encoding="utf-8")

split_table = pd.DataFrame({
    s: {"total": len(d), "label 0": int((d.label == 0).sum()), "label 1": int((d.label == 1).sum()),
        "synthetic": int(d.is_synthetic.sum())}
    for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]}).T
split_table

## Step 4 — Train the model
**Model:** TF-IDF on **character n-grams** + **Logistic Regression** (scikit-learn).

Why this model:
- Kazakh profanity appears in many spellings (Cyrillic/Latin, masked `с*к`, stretched `бооооқ`, Russian letters instead of Kazakh). Character n-grams (e.g. `боқ`, ` қот`, `*к`) match parts of words, so new spellings still share features with known ones.
- **Unknown words** never break the model: an unseen word is split into n-grams; n-grams not seen in training are simply ignored.
- **Punctuation** is removed by `normalize` (except `*`, which signals masking). **Long comments**: cut to 2,000 characters; TF-IDF with L2 normalization keeps long and short comments comparable.
- Small, fast (trains in seconds on CPU), and fully explainable: we can print the n-grams with the largest weights.
- `class_weight="balanced"` because profane comments are the minority class.

**How text becomes input:** raw text → `normalize()` → character n-grams inside word boundaries (`char_wb`) → TF-IDF vector (vocabulary and IDF **fitted on train only**) → logistic regression → probability P(profanity) → label 1 if probability ≥ threshold.

**Training objective:** logistic regression minimizes (class-weighted) binary cross-entropy / log-loss with L2 regularization (strength `1/C`).

**Model selection on validation:** small grid over n-gram range and `C`, choose the best validation F1 (profanity class); then choose the **decision threshold** that maximizes validation F1. The test set is not touched.

In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score

def build_model(ngram_range=(2, 5), C=4.0):
    return Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=normalize, analyzer="char_wb", ngram_range=ngram_range,
                                  min_df=2, sublinear_tf=True, max_features=200_000)),
        ("clf", LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEED)),
    ])

def best_threshold(y_true, prob):
    p, r, t = precision_recall_curve(y_true, prob)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    i = int(np.argmax(f1))
    return float(t[i]), float(f1[i])

X_train, y_train = train_df["text"], train_df["label"].values
X_val, y_val = val_df["text"], val_df["label"].values

grid = [(ng, C) for ng in [(1, 4), (2, 5), (3, 6)] for C in [0.5, 2.0, 8.0, 32.0]]
rows = []
for ng, C in grid:
    t0 = time.time()
    m = build_model(ng, C).fit(X_train, y_train)          # fit on TRAIN only
    prob = m.predict_proba(X_val)[:, 1]
    rows.append({"ngram_range": str(ng), "C": C,
                 "val_F1@0.5": f1_score(y_val, (prob >= 0.5).astype(int)),
                 "val_AP": average_precision_score(y_val, prob),
                 "seconds": round(time.time() - t0, 1)})
search = pd.DataFrame(rows).sort_values(["val_F1@0.5", "val_AP"], ascending=False).reset_index(drop=True)
search.to_csv("results/hyperparameter_search.csv", index=False)
search

In [ ]:
best = search.iloc[0]
BEST_NGRAM, BEST_C = eval(best["ngram_range"]), float(best["C"])

model = build_model(BEST_NGRAM, BEST_C).fit(X_train, y_train)
val_prob = model.predict_proba(X_val)[:, 1]
THRESHOLD, val_f1_thr = best_threshold(y_val, val_prob)
print(f"Chosen: ngram_range={BEST_NGRAM}, C={BEST_C}")
print(f"Validation F1 @0.5 = {f1_score(y_val, (val_prob >= 0.5).astype(int)):.3f} | "
      f"best threshold = {THRESHOLD:.3f} -> validation F1 = {val_f1_thr:.3f}")

# What did the model learn? n-grams with the biggest weights
vocab = np.array(model.named_steps["tfidf"].get_feature_names_out())
coef = model.named_steps["clf"].coef_[0]
print("\nTop profanity n-grams:", [repr(v) for v in vocab[np.argsort(coef)[-20:][::-1]]])
print("Top clean n-grams:    ", [repr(v) for v in vocab[np.argsort(coef)[:20]]])

In [ ]:
import joblib, sklearn
joblib.dump(model, "models/profanity_model.joblib")
config = {
    "model": "TF-IDF char_wb n-grams + LogisticRegression",
    "ngram_range": list(BEST_NGRAM), "C": BEST_C, "min_df": 2, "sublinear_tf": True,
    "class_weight": "balanced", "max_iter": 2000, "threshold": THRESHOLD, "seed": SEED,
    "preprocessing": "src/textprep.py: lowercase, ё->е, digits->0, mask symbols->*, collapse repeated letters, "
                     "remove punctuation, max 2000 chars",
    "split": "70/15/15 grouped by near-duplicate key, stratified by label",
    "split_sizes": {k: int(v) for k, v in split_table["total"].items()},
    "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
}
with open("models/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Saved models/profanity_model.joblib and models/config.json")
print(json.dumps(config, ensure_ascii=False, indent=2))

## Step 5 — Evaluate **once** on the held-out test set
All settings (n-grams, C, threshold) were chosen on validation. Now the test set is used one time.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

X_test, y_test = test_df["text"], test_df["label"].values
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= THRESHOLD).astype(int)

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
metrics = {
    "n_test": int(len(y_test)), "n_test_profane": int(y_test.sum()), "threshold": THRESHOLD,
    "precision_profane": precision_score(y_test, test_pred, zero_division=0),
    "recall_profane": recall_score(y_test, test_pred, zero_division=0),
    "f1_profane": f1_score(y_test, test_pred, zero_division=0),
    "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
}
# extra: real reviews only (synthetic examples can be easier)
real_mask = test_df["is_synthetic"].values == 0
if real_mask.sum() and y_test[real_mask].sum():
    metrics["real_only"] = {"n": int(real_mask.sum()),
                            "precision_profane": precision_score(y_test[real_mask], test_pred[real_mask], zero_division=0),
                            "recall_profane": recall_score(y_test[real_mask], test_pred[real_mask], zero_division=0),
                            "f1_profane": f1_score(y_test[real_mask], test_pred[real_mask], zero_division=0)}
with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(classification_report(y_test, test_pred, target_names=["0 clean", "1 profane"], digits=3, zero_division=0))
print(json.dumps(metrics, indent=2))

disp = ConfusionMatrixDisplay(cm, display_labels=["0 clean", "1 profane"])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Test confusion matrix (n={len(y_test)}, threshold={THRESHOLD:.2f})")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=150); plt.show()

**What the errors mean for review moderation**
- **False positive (FP):** a clean review is flagged as profane → an honest customer's review (often a useful *negative* review) is hidden. Too many FPs = censorship and lost trust.
- **False negative (FN):** a profane review is missed → offensive text is published on the product page.
- For moderation, **FN are usually worse** (offensive content visible to everyone), so recall on class 1 matters most. A practical setup is: lower the threshold a bit to raise recall, and send flagged reviews to a human moderator instead of deleting them automatically — that keeps the cost of FPs small.

## Step 6 — Inspect predictions and mistakes

In [ ]:
test_out = test_df[["text", "label", "is_synthetic", "label_source"]].copy()
test_out["prob_profane"] = test_prob.round(3)
test_out["pred"] = test_pred
test_out["lexicon_hits"] = test_out["text"].map(lambda t: ",".join(lexicon_hits(t)))
test_out["error"] = np.where(test_out.label == test_out.pred, "",
                             np.where(test_out.pred == 1, "FP", "FN"))
test_out.to_csv("results/test_predictions.csv", index=False, encoding="utf-8")

errors = test_out[test_out.error != ""].sort_values("prob_profane", ascending=False)
errors.to_csv("results/errors.csv", index=False, encoding="utf-8")
print(f"Test errors: {len(errors)}  (FP={int((errors.error=='FP').sum())}, FN={int((errors.error=='FN').sum())})")
errors.head(15)

In [ ]:
# Hand-written challenging examples (NOT in the dataset) to probe weaknesses
challenge = pd.DataFrame([
    ("Телефон керемет, бәріне ұсынамын!", 0, "clean positive review"),
    ("Сапасы өте нашар, ақшамды қайтарыңдар!!!", 0, "angry but clean"),
    ("Ауыр екен, көтеру қиын", 0, "substring trap: көтеру"),
    ("Амалсыз қайтардым, өлшемі келмеді", 0, "substring trap: амал"),
    ("Сатушы ақымақ, тауар басқа", 0, "mild insult (rule: 0)"),
    ("Мынау не боқ, екі күнде сынды", 1, "plain profanity"),
    ("б.о.қ телефон", 1, "letters separated by dots"),
    ("б о қ телефон", 1, "letters separated by spaces"),
    ("q0taq tauar", 1, "Latin + digit instead of o"),
    ("Курьер иттің баласы екен", 1, "curse formula"),
    ("Боктай сапа", 1, "Russian 'к' instead of 'қ'"),
], columns=["text", "expected", "case"])
challenge["prob_profane"] = model.predict_proba(challenge["text"])[:, 1].round(3)
challenge["pred"] = (challenge["prob_profane"] >= THRESHOLD).astype(int)
challenge["correct"] = challenge["pred"] == challenge["expected"]

sample_pred = pd.concat([
    test_out.sample(min(10, len(test_out)), random_state=SEED)[["text", "label", "pred", "prob_profane"]]
        .rename(columns={"label": "expected"}).assign(case="random test example"),
    challenge[["text", "expected", "pred", "prob_profane", "case"]],
], ignore_index=True)
sample_pred.to_csv("results/sample_predictions.csv", index=False, encoding="utf-8")
challenge

### Discussion of mistakes and challenging examples
Look at the two tables above (`errors` and `challenge`) and keep **at least three** cases in the README. Typical reasons you will see:

1. **Spaced / dotted letters (`б о қ`, `б.о.қ`)** — `normalize` turns dots into spaces and `char_wb` builds n-grams *inside* words, so the profane word disappears into single letters. *Fix:* join single letters separated by spaces/dots before vectorizing.
2. **Digit or Latin look-alikes (`q0taq`)** — digits become `0`, so `q0taq` ≠ `qotaq`. *Fix:* map look-alike characters (`0→o`, `3→з`, Latin→Cyrillic) in `normalize`, or add such variants to training data.
3. **Substring traps (`көтеру`, `амал`) and mild insults** — they share n-grams with profane words (`көт`, `ам`), so the probability rises. *Fix:* more real hard-negative examples; manual labeling of borderline lexicon hits.
4. **Lexicon-labeled real data** — if real labels come only from the lexicon, the model partly learns the lexicon and cannot be better than it on unseen words; test scores can look optimistic. *Fix:* manual review of `to_review.csv` (strongly recommended), more real profane comments.
5. **Synthetic examples are easier than real ones** — compare `real_only` metrics with the overall metrics in `results/metrics.json`.

## Step 7 — Predict on new text (no retraining)
Loads the saved model and threshold. Also writes `src/predict.py` for the command line:
```
python src/predict.py "Мынау не боқ, екі күнде сынды"
```

In [ ]:
%%writefile src/predict.py
import os, sys, json
import joblib

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)                    # textprep.py must be importable to load the model
ROOT = os.path.dirname(HERE)

_model = joblib.load(os.path.join(ROOT, "models", "profanity_model.joblib"))
with open(os.path.join(ROOT, "models", "config.json"), encoding="utf-8") as f:
    _threshold = json.load(f)["threshold"]


def predict(text: str) -> dict:
    prob = float(_model.predict_proba([text])[0, 1])
    label = int(prob >= _threshold)
    return {"text": text, "label": label, "prob_profane": round(prob, 4),
            "meaning": "Kazakh profanity present" if label else "no Kazakh profanity"}


if __name__ == "__main__":
    texts = sys.argv[1:] or [input("Enter a Kazakh comment: ")]
    for t in texts:
        r = predict(t)
        print(f"{r['label']}  (P(profanity)={r['prob_profane']:.3f}, threshold={_threshold:.3f})  {r['meaning']}  <- {t}")

In [ ]:
import joblib, json, sys
sys.path.insert(0, "src")
loaded_model = joblib.load("models/profanity_model.joblib")
loaded_threshold = json.load(open("models/config.json", encoding="utf-8"))["threshold"]

def predict(text):
    prob = float(loaded_model.predict_proba([text])[0, 1])
    label = int(prob >= loaded_threshold)
    print(f"Label: {label} ({'Kazakh profanity' if label else 'no Kazakh profanity'}) | "
          f"P(profanity) = {prob:.3f} | threshold = {loaded_threshold:.3f}")
    return label, prob

predict("Тауар сапалы, тез жеткізді, рахмет!")
predict("Мынау не боқ, ақшамды қайтар")

In [ ]:
# ✍️ Type your own comment here (change the text and run the cell)
my_text = "Телефон жақсы, бірақ зарядкасы тез бітеді"
predict(my_text)

In [ ]:
!python src/predict.py "Керемет сөмке, ұнады" "қ*тақ сапа, ешкімге алмаңдар"

## Step 8 — Write README, requirements and .gitignore, then zip the project
The README is filled automatically with the real numbers from this run. Afterwards **edit the sections marked ✍️** (your own discussion and contributions).

In [ ]:
import importlib.metadata as im
pkgs = ["pandas", "numpy", "scikit-learn", "matplotlib", "joblib", "datasets"]
with open("requirements.txt", "w") as f:
    for p in pkgs:
        try: f.write(f"{p}=={im.version(p)}\n")
        except im.PackageNotFoundError: f.write(p + "\n")
with open(".gitignore", "w") as f:
    f.write("data/all_reviews.csv\n__pycache__/\n.ipynb_checkpoints/\n*.zip\n")
print(open("requirements.txt").read())

In [ ]:
def md_table(df):
    df = df.copy()
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v).replace("|", "/").replace("\n", " ") for v in r) + " |"
                            for r in df.values)

counts = data.groupby(["is_synthetic", "label"]).size().unstack(fill_value=0)
n_real, n_syn = int((data.is_synthetic == 0).sum()), int(data.is_synthetic.sum())
ex = pd.concat([data[(data.label == 0) & (data.is_synthetic == 0)].sample(3, random_state=SEED),
                data[(data.label == 1) & (data.is_synthetic == 0)].head(2),
                data[data.is_synthetic == 1].sample(3, random_state=SEED)])[["text", "label", "is_synthetic", "label_source"]]
ex["text"] = ex["text"].str.slice(0, 100)
err_show = errors.head(5)[["text", "label", "pred", "prob_profane", "error"]].copy()
err_show["text"] = err_show["text"].str.slice(0, 100)
m = metrics
real_line = ""
if "real_only" in m:
    r = m["real_only"]
    real_line = f"\nReal reviews only (n={r['n']}): precision {r['precision_profane']:.3f}, recall {r['recall_profane']:.3f}, F1 {r['f1_profane']:.3f}.\n"
manual_used = int((data.label_source == "manual").sum())

readme = f'''# Kazakh Profanity Detection

DSSC5113 Algorithms of Computer Vision — Bonus Project (KBTU, Fall 2026)
**Author:** Zhanabil Alikhan · **Student ID:** 25MD0222

## 1. Task
Input: one Kazakh product review or comment. Output: **0 = no Kazakh profanity**, **1 = Kazakh profanity present**, together with the model probability P(profanity).

## 2. Repository structure
```
Kazakh_Profanity_Detection.ipynb   # full pipeline: data -> training -> evaluation -> prediction
src/textprep.py                    # text cleaning / normalization (used by the model)
src/lexicon.py                     # profanity patterns used to find and pre-label candidates
src/predict.py                     # command-line prediction with the saved model
data/labeled.csv                   # final labeled dataset
data/to_review.csv                 # real reviews for manual labeling
data/splits/train.csv val.csv test.csv
models/profanity_model.joblib      # trained model (TF-IDF + Logistic Regression pipeline)
models/config.json                 # settings, threshold, seed, versions
results/                           # metrics.json, confusion_matrix.png, test_predictions.csv, errors.csv, sample_predictions.csv
requirements.txt
```

## 3. Dataset
**Source:** KazSAnDRA — Kazakh Sentiment Analysis Dataset of Reviews and Attitudes (ISSAI, Nazarbayev University), CC BY 4.0.
https://huggingface.co/datasets/issai/kazsandra — downloaded automatically in Step 1 of the notebook (`data/all_reviews.csv`, not stored on GitHub because of size).

**Relation to the task:** real Kazakh user reviews of apps, products, books and places, including mixed Kazakh–Russian text, Latin script and slang — the kind of text a marketplace needs to moderate. Its sentiment labels do **not** match this task (negative ≠ profane), so they were not used; new profanity labels were created.

**Labeling method:**
1. Random sample of 2,000 reviews (seed {SEED}) + all reviews where the profanity lexicon (`src/lexicon.py`) found a candidate.
2. Lexicon gives a first label; labels checked by hand in `data/to_review.csv` → `data/manual_labels.csv`. Manually labeled rows in this run: **{manual_used}**. Rows without a manual check keep the lexicon label (`label_source = lexicon_auto`).
3. **Synthetic examples** (`is_synthetic = 1`, `label_source = synthetic_template`): template comments written by the author, with matching clean negative templates (same sentence, no profanity) and substring traps (*көтеру*, *амал*, *бокс*).

**Size:** {len(data)} samples — {n_real} real, {n_syn} synthetic. Class distribution: label 0 = {int((data.label==0).sum())}, label 1 = {int((data.label==1).sum())}.

| | label 0 | label 1 |
|---|---|---|
| real | {counts.loc[0, 0] if 0 in counts.index else 0} | {counts.loc[0, 1] if 0 in counts.index and 1 in counts.columns else 0} |
| synthetic | {counts.loc[1, 0] if 1 in counts.index else 0} | {counts.loc[1, 1] if 1 in counts.index and 1 in counts.columns else 0} |

Personal information (URLs, e-mails, phone numbers, @usernames) is removed; no reviewer names are stored.

### Labeling rules
- **1** — obscene/vulgar Kazakh words (sexual, genital, excretory: e.g. *боқ*, *қотақ*, *сік-*, *жезөкше*, *көтіңе*) and fixed curse formulas (*әкеңнің аузын…*, *иттің баласы*).
- **1** — masked or misspelled forms with clear meaning (*с\\*к*, *б\\*\\*қ*, *бооооқ*, Latin *qotaq*).
- **1** — Kazakh profanity inside a mixed Kazakh–Russian comment.
- **0** — negative or angry reviews without profanity (*Сапасы өте нашар, ақшамды қайтарыңыз*).
- **0** — mild, non-obscene insults (*ақымақ*, *топас*, *оңбаған*).
- **0** — words that only contain a profane-looking substring (*көтеру*, *амал*, *бокс*).
- **0** — profanity only in Russian (out of scope: the task is *Kazakh* profanity) — listed as a limitation.
- Ambiguous cases → 0, with a comment in the `note` column.

### Labeled examples
{md_table(ex)}

### Splits
Grouped split (near-duplicates in the same split): real reviews are grouped by their letters-only normalized text; synthetic examples by their (template, word) family. 70/15/15, stratified, **seed = {SEED}**. Exact duplicates removed.

{md_table(split_table.reset_index().rename(columns={"index": "split"}))}

## 4. Model
**TF-IDF character n-grams (`char_wb`, n-gram range {BEST_NGRAM}) + Logistic Regression (C = {BEST_C}, class_weight = balanced)**, scikit-learn.

- *Why:* profanity appears in many spellings (Cyrillic/Latin, masked, stretched, Russian letters instead of Kazakh). Character n-grams share features between spelling variants; the model is small, fast, and interpretable (largest-weight n-grams can be printed).
- *Text → input:* raw text → `normalize()` (lowercase, ё→е, digits→0, mask symbols→`*`, repeated letters collapsed, punctuation removed, max 2,000 chars) → character n-grams → TF-IDF vector (vocabulary and IDF fitted on **train only**) → logistic regression → P(profanity) → label 1 if P ≥ threshold.
- *Long comments:* cut to 2,000 characters; sublinear TF + L2 normalization keep long and short texts comparable.
- *Punctuation:* removed, except `*` which marks masking.
- *Unknown words:* split into character n-grams; n-grams never seen in training are ignored, so the model never fails on new words.
- *Objective:* class-weighted binary cross-entropy (log-loss) with L2 regularization.
- *Selection:* grid over n-gram range {{(1,4),(2,5),(3,6)}} × C {{0.5, 2, 8, 32}} by validation F1; decision **threshold = {THRESHOLD:.3f}** chosen to maximize validation F1 (`results/hyperparameter_search.csv`).

## 5. Setup and usage
```bash
git clone <this repo>
cd <repo>
pip install -r requirements.txt
```
**Train from scratch (reproducible):** open `Kazakh_Profanity_Detection.ipynb` (locally or in Google Colab) → *Run all*. Seed = {SEED}.

**Predict without retraining:**
```bash
python src/predict.py "Мынау не боқ, екі күнде сынды"
```
or in Python:
```python
import sys; sys.path.insert(0, "src")
from predict import predict
print(predict("Тауар сапалы, рахмет!"))
```
The trained model (`models/profanity_model.joblib`) and its preprocessing (`src/textprep.py`, `models/config.json`) are in this repository.

## 6. Results (test set, evaluated once)
Test examples: **{m['n_test']}** ({m['n_test_profane']} profane). Threshold {THRESHOLD:.3f}.

| Metric (profanity class) | Value |
|---|---|
| Precision | {m['precision_profane']:.3f} |
| Recall | {m['recall_profane']:.3f} |
| F1 | {m['f1_profane']:.3f} |
{real_line}
Confusion matrix (rows = true, columns = predicted): TN = {m['confusion_matrix']['TN']}, FP = {m['confusion_matrix']['FP']}, FN = {m['confusion_matrix']['FN']}, TP = {m['confusion_matrix']['TP']}

![confusion matrix](results/confusion_matrix.png)

- **False positive:** a clean review is flagged → an honest (often negative but useful) review gets hidden.
- **False negative:** a profane review is missed → offensive text is published.
- For moderation, false negatives are usually worse; flagged reviews should go to a human moderator, and the threshold can be lowered to increase recall.

### Errors and challenging examples
Test errors (first 5, full list in `results/errors.csv`):

{md_table(err_show) if len(err_show) else "No test errors in this run - see challenging examples below."}

Hand-written challenging examples (`results/sample_predictions.csv`):

{md_table(challenge[["text", "expected", "pred", "prob_profane", "case"]])}

✍️ **Discussion (edit after looking at the tables):**
1. *Spaced/dotted letters* (`б о қ`): `char_wb` builds n-grams inside words, so the profane word is split into single letters. Fix: join single letters before vectorizing.
2. *Look-alike characters* (`q0taq`): digits are mapped to `0`, so the variant does not match `qotaq`. Fix: map look-alikes (0→o, Latin→Cyrillic) in `normalize` or add variants to training data.
3. *Substring traps and mild insults* (`көтеру`, `ақымақ`): share n-grams with profane words. Fix: more real hard negatives and manual review of borderline cases.

## 7. Limitations
- Real profane reviews are rare; many positives are synthetic, which are easier than real comments (compare overall vs real-only metrics).
- Real labels not checked by hand come from the lexicon, so the model partly learns the lexicon and test scores can be optimistic on lexicon-like words.
- Russian-only profanity is labeled 0 by design; the model is not a general toxicity detector.
- Small lexicon; regional slang and new spellings may be missed.

## 8. Acknowledgments
Dataset: KazSAnDRA (Yeshpanov & Varol, LREC-COLING 2024), CC BY 4.0.
AI assistance: Claude (Anthropic) helped draft the notebook code and README. ✍️ My own contributions: (fill in — e.g. labeling rules, manual labeling, lexicon review, error analysis, experiments).
'''
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print("README.md written,", len(readme), "characters")

In [ ]:
# Zip the whole project so you can download it and upload it to GitHub
import shutil, os
exclude = {"data/all_reviews.csv"}
os.makedirs("/tmp/kpd_export", exist_ok=True)
export = "/tmp/kpd_export/kazakh-profanity-detection"
if os.path.exists(export): shutil.rmtree(export)
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"__pycache__", "sample_data"}]
    for fn in files:
        rel = os.path.normpath(os.path.join(root, fn))
        if rel.replace("\\", "/") in exclude or rel.endswith(".zip"): continue
        if not (rel.endswith(".ipynb") or rel.split(os.sep)[0] in {"src", "data", "models", "results"}
                or rel in {"README.md", "requirements.txt", ".gitignore"}): continue
        os.makedirs(os.path.join(export, os.path.dirname(rel)), exist_ok=True)
        shutil.copy2(rel, os.path.join(export, rel))
shutil.make_archive("kazakh-profanity-detection", "zip", export)
print("Created kazakh-profanity-detection.zip")
try:
    from google.colab import files       # in Colab: download automatically
    files.download("kazakh-profanity-detection.zip")
except ImportError:
    pass